# ICU LOS Classification Demo

This notebook demonstrates the saved ICU length-of-stay classifier using a small synthetic dataset. The sample data match the saved model's feature schema, but they do not contain real MIMIC-IV patient records. The real training data are restricted and are not included in this submission.

In [5]:
from pathlib import Path
import contextlib
import io
import os
import warnings

# This avoids a noisy joblib/loky CPU-count warning on some macOS setups.
os.environ.setdefault("LOKY_MAX_CPU_COUNT", "8")
warnings.filterwarnings("ignore", message="Could not find the number of physical cores.*")

import joblib
import pandas as pd
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, f1_score

MODEL_FILE = Path("models/icu_los_classifier.joblib")
SAMPLE_FILE = Path("data/sample/icu_los_classification_sample.csv")
REQUIRED_ARTIFACT_KEYS = ["model", "feature_columns", "target_column", "target_labels"]


def find_submission_root() -> Path:
    """Find the folder that contains the demo model and sample data."""
    cwd = Path.cwd()
    candidates = []

    if cwd.name == "submission":
        candidates.append(cwd)
    if cwd.name == "notebooks":
        candidates.append(cwd.parent)

    # These make the notebook work when opened from either the full repo root
    # or the submission folder itself.
    candidates.extend([cwd / "submission", cwd.parent / "submission", cwd])

    for folder in candidates:
        if (folder / MODEL_FILE).exists() and (folder / SAMPLE_FILE).exists():
            return folder

    raise FileNotFoundError(
        "Could not find models/icu_los_classifier.joblib and "
        "data/sample/icu_los_classification_sample.csv."
    )


def run_quietly(function, *args):
    """Run model loading/prediction while hiding local joblib diagnostics."""
    stderr_buffer = io.StringIO()
    with contextlib.redirect_stderr(stderr_buffer):
        return function(*args)


def load_model_artifact(model_path: Path) -> dict:
    """Load the saved model bundle and check the fields used by the demo."""
    artifact = run_quietly(joblib.load, model_path)
    missing_keys = [key for key in REQUIRED_ARTIFACT_KEYS if key not in artifact]
    if missing_keys:
        raise KeyError(f"Model artifact is missing expected keys: {missing_keys}")
    return artifact


def load_demo_sample(sample_path: Path, feature_columns: list[str], target_column: str) -> pd.DataFrame:
    """Load the synthetic sample data and verify it matches the model schema."""
    sample_df = pd.read_csv(sample_path)
    required_columns = ["subject_id", "stay_id", target_column] + list(feature_columns)
    missing_columns = [column for column in required_columns if column not in sample_df.columns]
    if missing_columns:
        raise ValueError(f"Sample data is missing required columns: {missing_columns[:10]}")
    return sample_df


def predict_los_categories(model, features: pd.DataFrame):
    """Generate class predictions and class probabilities."""
    predictions = run_quietly(model.predict, features)
    probabilities = run_quietly(model.predict_proba, features)
    return predictions, probabilities


In [6]:
ROOT = find_submission_root()
artifact = load_model_artifact(ROOT / MODEL_FILE)

model = artifact["model"]
feature_columns = artifact["feature_columns"]
target_column = artifact["target_column"]
target_labels = artifact["target_labels"]

sample = load_demo_sample(ROOT / SAMPLE_FILE, feature_columns, target_column)
X = sample[feature_columns]
y = sample[target_column]

pred, proba = predict_los_categories(model, X)

print(f"Loaded {len(sample)} synthetic demo rows")
print(f"Model expects {len(feature_columns)} features")
print("Target labels:", target_labels)


Loaded 24 synthetic demo rows
Model expects 380 features
Target labels: {0: 'LOS < 2 days', 1: '2 <= LOS <= 7 days', 2: 'LOS > 7 days'}


In [7]:
def build_prediction_table(sample_df: pd.DataFrame, predictions, probabilities) -> pd.DataFrame:
    """Create a small readable table of prediction examples."""
    results_df = sample_df[["subject_id", "stay_id", target_column]].copy()
    results_df["predicted_los_category"] = predictions

    for index, class_label in enumerate(model.classes_):
        results_df[f"prob_class_{class_label}"] = probabilities[:, index]

    return results_df


results = build_prediction_table(sample, pred, proba)

print("Macro F1:", round(f1_score(y, pred, average="macro", zero_division=0), 3))
print("Weighted F1:", round(f1_score(y, pred, average="weighted", zero_division=0), 3))
print("Balanced accuracy:", round(balanced_accuracy_score(y, pred), 3))
print("\nConfusion matrix, rows=true and columns=predicted:")
print(confusion_matrix(y, pred, labels=[0, 1, 2]))

results.head(10)


Macro F1: 0.627
Weighted F1: 0.63
Balanced accuracy: 0.653

Confusion matrix, rows=true and columns=predicted:
[[5 1 0]
 [3 5 0]
 [2 3 5]]


,subject_id,stay_id,los_category,predicted_los_category,prob_class_0,prob_class_1,prob_class_2
0,900000,700000,0,0,0.572021,0.419726,0.008253
1,900001,700001,0,0,0.865641,0.133699,0.000661
2,900002,700002,0,0,0.922793,0.076581,0.000626
3,900003,700003,0,0,0.951184,0.048003,0.000813
4,900004,700004,0,0,0.943102,0.055232,0.001667
5,900005,700005,1,1,0.444586,0.542157,0.013257
6,900006,700006,1,1,0.326562,0.669867,0.003570
7,900007,700007,1,1,0.275516,0.719301,0.005183
8,900008,700008,1,1,0.406980,0.582839,0.010181
9,900009,700009,1,1,0.360660,0.630135,0.009205


In [8]:
class_names = [target_labels[class_id] for class_id in [0, 1, 2]]
print(classification_report(y, pred, labels=[0, 1, 2], target_names=class_names, zero_division=0))


                    precision    recall  f1-score   support

      LOS < 2 days       0.50      0.83      0.62         6
2 <= LOS <= 7 days       0.56      0.62      0.59         8
      LOS > 7 days       1.00      0.50      0.67        10

          accuracy                           0.62        24
         macro avg       0.69      0.65      0.63        24
      weighted avg       0.73      0.62      0.63        24



## Real Held-Out Test Interpretation

The table above is based on the synthetic demo sample, which is only included to show that the saved model can run. The main performance interpretation should come from the real held-out test set reported in `models/icu_los_classifier_metadata.json` and the full project reports. On that real test set, the model performs best on short ICU stays (`<2` days). It has more difficulty separating intermediate and long stays. For long stays (`>7` days), recall is moderate but precision is lower, so the model tends to over-flag some shorter stays as prolonged.

In [9]:
results_with_errors = results.copy()
results_with_errors["correct"] = results_with_errors[target_column].eq(
    results_with_errors["predicted_los_category"]
)

results_with_errors.sort_values("correct").head(10)


,subject_id,stay_id,los_category,predicted_los_category,prob_class_0,prob_class_1,prob_class_2,correct
23,900023,700023,0,1,0.286138,0.707932,0.005930,False
21,900021,700021,2,0,0.362949,0.354639,0.282412,False
20,900020,700020,2,1,0.112392,0.594195,0.293413,False
19,900019,700019,2,1,0.111982,0.645688,0.242330,False
18,900018,700018,2,1,0.288007,0.444755,0.267239,False
17,900017,700017,1,0,0.919134,0.079519,0.001347,False
16,900016,700016,1,0,0.557567,0.434589,0.007844,False
15,900015,700015,1,0,0.949099,0.044265,0.006636,False
22,900022,700022,2,0,0.433327,0.397313,0.169360,False
14,900014,700014,2,2,0.160767,0.415480,0.423754,True
